### 创建智能体

1. 安装依赖(包括LangChain、LangGraph、DeepAgents)
2. 初始化LLM
3. 初始化工具列表
4. 初始化智能体
5. 测试智能体

1. 安装依赖

```
pip install -U langchain langgraph deepagents
```

目前langchain和langgraph要求python版本号<strong>3.10及3.10+</strong>,
deepagents要求python版本号<strong>3.10+</strong>,
所以推荐在python=3.11环境下进行相关内容的学习

In [11]:
%pip install -U langchain langchain_openai langgraph deepagents

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached langchain-1.4.5-py3-none-any.whl (166 kB)
     ---------------------------------------- 0.0/573.3 kB ? eta -:--:--
     ------------------ --------------------- 262.1/573.3 kB ? eta -:--:--
     ---------------------------------------- 573.3/573.3 kB 1.8 MB/s  0:00:00
     ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
     --------- ------------------------------ 0.5/2.2 MB 4.2 MB/s eta 0:00:01
     ------------------- -------------------- 1.0/2.2 MB 3.3 MB/s eta 0:00:01
     --------------------------------- ------ 1.8/2.2 MB 3.5 MB/s eta 0:00:01
     ---------------------------------------- 2.2/2.2 MB 3.1 MB/s  0:00:00
     ---------------------------------------- 0.0/944.4 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/944.4 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/944.4 kB ? eta -:--:--
     ---------------------------------------- 944.4/

2. 初始化LLM

通常在本地环境或.env配置LLM相关参数

- 在本地环境进行配置

**Windows**环境下：

```bash
# 临时(当前终端窗口生效)
set MODEL_NAME=deepseek-flash
set MODEL_API_KEY=sk-...
set MODEL_API_URL=https://api.deepseek.com/v1

# 永久(powershell，写入用户环境变量，重启终端生效)
[Environment]::SetEnvironmentVariable("MODEL_NAME", "deepseek-flash", "User")
[Environment]::SetEnvironmentVariable("MODEL_API_KEY", "sk-...", "User")
[Environment]::SetEnvironmentVariable("MODEL_API_URL", "https://api.deepseek.com/v1", "User")
```

**Linux**环境下：

```bash
# 临时
export MODEL_NAME="deepseek-flash"
export MODEL_API_KEY="sk-..."
export MODEL_API_URL="https://api.deepseek.com/v1"

# 永久
echo 'export MODEL_NAME="deepseek-flash"' >> ~/.bashrc
echo 'export MODEL_API_KEY="sk-..."' >> ~/.bashrc
echo 'export MODEL_API_URL="https://api.deepseek.com/v1"' >> ~/.bashrc
source ~/.bashrc
```

- 使用.env文件进行配置

```text
# 示例
MODEL_NAME="deepseek-flash"
MODEL_API_KEY=sk-...
MODEL_API_URL=https://api.deepseek.com/v1
```

**读取环境变量**

```python
import os
# %pip install python_dotenv
from dotenv import load_dotenv

load_dotenv()

model_name = os.getenv("MODEL_NAME")
api_key = os.getenv("MODEL_API_KEY")
api_url = os.getenv("MODEL_API_URL")
```

In [3]:
# 在Windows环境下临时配置LLM相关参数
# %env是jupyter原生魔法，Winodws终端环境下应该使用set
%env MODEL_NAME=deepseek-flash
# 需要替换为真实API Key
%env MODEL_API_KEY=sk-c5d6c79c0c1448fba710294d32106c04
%env MODEL_API_URL=https://api.deepseek.com/v1

env: MODEL_NAME=deepseek-flash
env: MODEL_API_KEY=sk-c5d6c79c0c1448fba710294d32106c04
env: MODEL_API_URL=https://api.deepseek.com/v1


In [5]:
%pip install dotenv

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

   ---------------------------------------- 2/2 [dotenv]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model_name = os.getenv("MODEL_NAME")
model_api_key = os.getenv("MODEL_API_KEY")
model_api_url = os.getenv("MODEL_API_URL")

# 其他参数可以从https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model查阅
chat_model = init_chat_model(
    model=model_name,
    model_provider="openai", # deepseek兼容openai协议
    api_key=model_api_key,
    base_url=model_api_url,
    temperature=0.1 #随机性，越接近1随机性越强
)

# 测试
# chat_model 输入消息列表
ai_reply = chat_model.invoke([
        {
            "role":"user",
            "content":"你好"
        }
    ])

print(ai_reply)
print(ai_reply.content)

ModuleNotFoundError: No module named 'langgraph.chat_models'

3. 初始化工具列表

工具重要参数：

- name
- description
- args_schema

**@tool**作用：

1. 生成tool description

将函数的文档字符串""" """变为工具描述。数据类型是string。

2. 生成tool args_schema

读取函数的type hint,生成符合OpenAI function calling的参数结构。

3. 将普通函数变为Base Tool对象

包装后函数拥有.name、.description、.args_schema属性，传给bind_tools或者create_agent。

4. 适配LangGraph的ToolNode

LangGraph中的ToolNode会自动识别@tool装饰的工具，
自动执行、把结果包装成ToolMessage,
不需要处理消息转换

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def get_weather(city: str) -> str:
    """返回该城市的天气情况
    
    Args:
        city: 城市名

    Returns:
        str: 该城市的天气情况，例如"晴天"
    """
    return f"{city}当前天气：晴天"

chat_model_with_tools = chat_model.bind_tools([get_weather])

res = chat_model_with_tools.invoke([HumanMessage(content="贵阳天气怎么样？")])

print("res.tool_calls:", res.tool_calls)

res.tool_calls: [{'name': 'get_weather', 'args': {'city': '贵阳'}, 'id': 'call_00_wWnkWpK1d7kmhGNyTLOs7720', 'type': 'tool_call'}]


**chat_model**通过
*bind_tools*绑定tool list后，当识别到存在工具调用的需求，会返回tool_calls表示需要调用工具，chat_model没有工具调用的能力，因为其只是LLM，不是Agent

4. 初始化智能体

不同于
**chat_model**,
**agent**拥有了工具调用的能力,遵守ReAct模式,
即reasoning->action->observation的不断循环，
具体可查阅论文[ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)

In [28]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=chat_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

In [31]:
# 5. 测试智能体

res = agent.invoke(
    {
        "messages":[
            HumanMessage(content="贵阳天气怎么样？")
        ]
    }
)

print(res["messages"][-1].content_blocks)

[{'type': 'text', 'text': '贵阳当前天气是**晴天** ☀️，适合外出活动哦！'}]
